In [3]:
from collections import defaultdict
import json
import tqdm

def get_interaction(datas):
    user_seq = {}
    user_seq_notime = {}
    for data in datas:
        user, item, time = data
        if user in user_seq:
            if item not in user_seq_notime[user]:
                user_seq[user].append((item, time))
                user_seq_notime[user].append(item)
            else:
                continue
        else:
            user_seq[user] = []
            user_seq_notime[user] = []
            user_seq[user].append((item, time))
            user_seq_notime[user].append(item)
    for user, item_time in user_seq.items():
        item_time.sort(key=lambda x: x[1])
        items = []
        for t in item_time:
            items.append(t[0])
        user_seq[user] = items
    return user_seq

def check_Kcore(user_items, user_core, item_core):
    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item] += 1
    for user, num in user_count.items():
        if num < user_core:
            return user_count, item_count, False
    for item, num in item_count.items():
        if num < item_core:
            return user_count, item_count, False
    return user_count, item_count, True

def filter_Kcore(user_items, user_core, item_core):
    user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    while not isKcore:
        for user, num in user_count.items():
            if user_count[user] < user_core:
                user_items.pop(user)
            else:
                for item in user_items[user]:
                    if item_count[item] < item_core:
                        user_items[user].remove(item)
        user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    return user_items

def load_artist_names():
    artist_id_to_name = {}
    artist_file = './artists.dat'  # 假设 artists.dat 在当前目录
    with open(artist_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[1:]  # 跳过表头
        for line in lines:
            parts = line.strip().split('\t')
            artist_id, artist_name = parts[0], parts[1]  # id 和 name
            artist_id_to_name[artist_id] = artist_name
    return artist_id_to_name

def id_map(user_items, artist_id_to_name):
    user2id = {}
    item2id = {}
    id2user = {}
    id2item = {}  # 将存储 item_id 到 artist_name 的映射
    user_id = 1
    item_id = 1
    final_data = {}
    for user, items in user_items.items():
        if user not in user2id:
            user2id[user] = str(user_id)
            id2user[str(user_id)] = user
            user_id += 1
        iids = []
        for item in items:  # item 是 artistID
            if item not in item2id:
                item2id[item] = str(item_id)
                # 使用 artist_id_to_name 获取艺术家名称，若无则用 artistID
                id2item[str(item_id)] = artist_id_to_name.get(item, item)
                item_id += 1
            iids.append(item2id[item])
        uid = user2id[user]
        final_data[uid] = iids
    data_maps = {
        'user2id': user2id,
        'item2id': item2id,
        'id2user': id2user,
        'id2item': id2item
    }
    return final_data, user_id - 1, item_id - 1, data_maps

def LastFM():
    user_core = 5
    item_core = 5
    datas = []
    data_file = './user_taggedartists-timestamps.dat'  # 数据文件路径
    lines = open(data_file).readlines()
    for line in tqdm.tqdm(lines[1:]):  # 跳过表头
        user, item, attribute, timestamp = line.strip().split('\t')
        datas.append((user, item, int(timestamp)))

    # 加载艺术家名称
    artist_id_to_name = load_artist_names()

    user_items = get_interaction(datas)
    user_items = filter_Kcore(user_items, user_core=user_core, item_core=item_core)
    user_items, user_num, item_num, data_maps = id_map(user_items, artist_id_to_name)

    # 生成 LastFM.txt (每行一个 user_id 和 item_id 对)
    with open('LastFM.txt', 'w') as f:
        for user, items in user_items.items():
            for item in items:
                f.write(f"{user} {item}\n")
    
    # 生成 LastFM_id2name.txt (item_id::item_title)
    with open('LastFM_id2name.txt', 'w') as f:
        for item_id, item_title in data_maps['id2item'].items():
            f.write(f"{item_id}::{item_title}\n")

LastFM()

100%|██████████| 186479/186479 [00:00<00:00, 475977.25it/s]
